In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
import snspd
params = snspd.snspd()
# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260413-26396-qcodes.log
Experiment loaded. Last ID no: 464


In [ ]:
verticalsnap = snap['instruments']['mso5']['submodules']['channels']['channels']['mso5_ch1']['parameters'].keys()

In [ ]:
horizontalsnap = snap['instruments']['mso5']['parameters'].keys()

In [3]:
station = Station(config_file="friesland.yaml")

In [3]:
yoko = station.load_instrument("yoko", revive_instance=True)

Connected to: YOKOGAWA GS210 (serial:91T928105, firmware:2.02) in 0.09s


In [4]:
laser = station.load_instrument("laser", revive_instance=True)

2026-04-13 16:27:16,313 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ D:\SNSPD\SNSPD2\PPCL550.py:4: QCoDeSDeprecationWarning: The `qcodes.utils.helpers` module is deprecated. Please consult the api documentation at https://microsoft.github.io/Qcodes/api/index.html for alternatives.
  from qcodes.utils.helpers import create_on_off_val_mapping



Connected to: PurePhotonic PPCL550 (serial:PP70AJ005, firmware:PV 2.0.0:HW 3.0.0:FW 7.0.0:AS C1:OT 1.0.0) in 1.62s


In [4]:
yoko.current()

6e-06

In [18]:
params = snspd4.snspd4()

In [11]:
snap = station.snapshot(update=True)

In [12]:
horizontalsnap = snap['instruments']['mso5']['parameters'].keys()

KeyError: 'mso5'

In [13]:
load_by_id(params.system_dark_counts_id).snapshot['station']['instruments']['mso5']['parameters']['horizontal_samplerate']

{'__class__': 'qcodes.parameters.parameter.Parameter',
 'full_name': 'mso5_horizontal_samplerate',
 'value': 625000000.0,
 'raw_value': '625.0000E+6',
 'ts': '2026-04-10 18:58:58',
 'instrument': 'MSO5.MSO5',
 'instrument_name': 'mso5',
 'name': 'horizontal_samplerate',
 'label': 'horizontal_samplerate',
 'validators': [],
 'inter_delay': 0,
 'unit': 'Sa/s',
 'post_delay': 0}

In [14]:
load_by_id(params.att_screw_calibration_id).snapshot['station']['instruments']['mso5']['parameters']['horizontal_samplerate']

KeyError: 'mso5'

In [16]:
initial = 8e-6
currents = np.arange(6e-6, 12e-6, 0.5e-6)
start = currents[0]
if initial >= start: 
    ramp = np.arange(initial, start-0.1e-6, -0.1e-6) # ramp down 
elif initial < start: 
    ramp = np.arange(initial, start+0.1e-6, 0.1e-6) # ramp up

for r in ramp: 
    curr_set = round(r, 9)
    print(curr_set)
    time.sleep(0.5)


8e-06
7.9e-06
7.8e-06
7.7e-06
7.6e-06
7.5e-06
7.4e-06
7.3e-06
7.2e-06
7.1e-06
7e-06
6.9e-06
6.8e-06
6.7e-06
6.6e-06
6.5e-06
6.4e-06
6.3e-06
6.2e-06
6.1e-06
6e-06


In [9]:
r = 5.69
round(r, 2)

5.69

In [ ]:
def current_sweep(yoko, dmm, currents, station=None):
    
    # # Update experiment snapshot 
    # update_station(station)
    
    # Ramping to start current 
    start = currents[0]
    if yoko.current() >= start: 
        ramp = np.arange(initial, start-0.1e-6, -0.1e-6) # ramp down 
    elif yoko.current() < start: 
        ramp = np.arange(initial, start+0.1e-6, 0.1e-6) # ramp up

    print(f'Ramping current from {yoko.current()}A to {start}A')

    for r in ramp: 
        yoko.current(round(r, 9)) # Set the current 
        time.sleep(0.5)
    
    # Initialize measurement 
    meas = Measurement()
    meas.register_custom_parameter("ratio")
    meas.register_parameter(yoko.current)
    meas.register_parameter(dmm.volt)
    # meas.register_custom_parameter("T_MXC", label="mK")


    with meas.run() as datasaver:
        print(datasaver.run_id)
        for i in currents:
            yoko.current(i)
            time.sleep(2)
            if i == 0: 
                ratio = 0
            else: 
                ratio = dmm.volt()/yoko.current()
            
            # if tc.channels['MXC-flange'].measure()['temperature'][-1] == None:
            #     temp = -1
            # else: 
            #     temp = tc.channels['MXC-flange'].measure()['temperature'][-1]*1e3
            
            datasaver.add_result(('ratio',ratio),
                                (dmm.volt, dmm.volt()),
                                (yoko.current, yoko.current()))
                                # ("T_MXC", temp))
            time.sleep(0.1)

    # Ramping down
    print('Ramping down')
    for i in currents[::-1]: 
        yoko.current(i)
        time.sleep(0.1)

    print(f'Current is {yoko.current()}')
    print('Finished!')


In [5]:
from functions import laser_set_standard, laser_get_standard

In [6]:
laser.frequency_coarse()

193414400000000.0

In [7]:
laser.frequency_fine()

0.0

In [8]:
laser_set_standard(laser, 1528e-9, 7)

In [9]:
laser.frequency_coarse()

196199200000000.0

In [6]:
from functions import laser_set_standard, laser_get_standard
# wav_range = np.arange(1528e-9, 1565e-9, 1e-9)
wav_range = np.arange(1528e-9, 1535e-9, 1e-9)
for l in wav_range: 
    laser_set_standard(laser, l, 7)
    time.sleep(1)
    laser_get_standard(laser)
    print('\n')

Power: 7.0
Frequency coarse: 196.1992THz
Wavelength (calculated) is 1528.0004097876035nm


Power: 7.0
Frequency coarse: 196.0709THz
Wavelength (calculated) is 1529.0002647001672nm


Power: 7.0
Frequency coarse: 195.9427THz
Wavelength (calculated) is 1530.000648148668nm


Power: 7.0
Frequency coarse: 195.8147THz
Wavelength (calculated) is 1531.0007777761323nm


Power: 7.0
Frequency coarse: 195.6869THz
Wavelength (calculated) is 1532.0006500179622nm


Power: 7.0
Frequency coarse: 195.5593THz
Wavelength (calculated) is 1533.0002613018148nm


Power: 7.0
Frequency coarse: 195.4318THz
Wavelength (calculated) is 1534.0003929759641nm




# Counts vs Wavelength

In [ ]:
def snspd_counts_vs_wavelength(MS, dmm, yoko, p_att, laser, device_name, n_captures, interval, wavelength_range, station=None):
    '''
    interval is specified in seconds
    '''

    # Update station
    update_station(station)

    # Establish measurement
    meas = Measurement()
    meas.register_parameter(dmm.volt)
    meas.register_parameter(yoko.current)
    meas.register_custom_parameter("threshold1", label="V")
    meas.register_custom_parameter("threshold2", label="V")
    meas.register_custom_parameter("total_counts1", label="counts")
    meas.register_custom_parameter("total_counts2", label="counts")
    meas.register_custom_parameter("counts1")
    meas.register_custom_parameter("counts2")
    meas.register_custom_parameter("trace_time", label="s")
    meas.register_custom_parameter("meas_time", label="s")
    meas.register_custom_parameter("interval", label="s")
    meas.register_custom_parameter("CR1", label="cps")
    meas.register_custom_parameter("CR2", label="cps")
    meas.register_custom_parameter("n_captures")
    meas.register_custom_parameter("v_attenuator", label="V")
    meas.register_custom_parameter("wavelength", label="nm")



    with meas.run() as datasaver:
        print(datasaver.run_id)
        
        # save device name 
        datasaver.dataset.add_metadata("device", device_name)
    
    
        # Set current
        yoko.current(current)
        time.sleep(1)
        print(f'Current is {yoko.current()}')

        if MS.channels[0].clipping(): 
            print('Error: Clipping')

        for wav in wavelength_range: # <- sweep wavelength of laser 

            # Set laser wavelength
            laser.frequency_coarse(spc.c/wavelength)
            time.sleep(10)
            laser_get_standard(laser)
            print('\n')


            # Extract the amount of time in one trace 
            h_time = MS.horizontal_scale()*MS.horizontal_divisions()
    
            
            # Set thresholds based on current 
            threshold1, threshold2 = set_thresholds(current)

            # Set thresholds  
            MS.write(f'SEARCH:SEARCH1:TRIGger:A:EDGE:THReshold {threshold1}')
            MS.write(f'SEARCH:SEARCH2:TRIGger:A:EDGE:THReshold {threshold2}')

            time.sleep(5)

            # Run count 
            MS.write("SEARCH:SEARCH1:STATE 0")
            MS.write("SEARCH:SEARCH1:STATE 1")
            MS.write("SEARCH:SEARCH2:STATE 0")
            MS.write("SEARCH:SEARCH2:STATE 1")

            start = time.perf_counter()
            print(f'This acquisition will take {n_captures*interval}s')
            print(datetime.datetime.now().hour,  datetime.datetime.now().minute)

            counts1= []
            counts2= []

            
            for i in range(n_captures):
                time.sleep(interval)

                # Extract counts 
                count1 = int(MS.ask("SEARCH:SEARCH1:TOTal?"))
                count2 = int(MS.ask("SEARCH:SEARCH2:TOTal?"))

                counts1.append(count1)
                counts2.append(count2)

                
            # calculate total counts 
            total_counts1 = sum(counts1)
            total_counts2 = sum(counts2)
            
            # total time in measurement 
            meas_time = n_captures*h_time
            
            # dark count rate calculation
            CR1 = total_counts1/meas_time
            CR2 = total_counts2/meas_time
            
            # Save data 
            datasaver.add_result((yoko.current, yoko.current()),
                                (dmm.volt, dmm.volt()),
                                ("threshold1", threshold1), 
                                ("threshold2", threshold2), 
                                ("total_counts1", total_counts1), 
                                ("total_counts2", total_counts2), 
                                ("counts1", counts1), 
                                ("counts2", counts2), 
                                ("meas_time", meas_time), 
                                ("interval", interval), 
                                ("n_captures", n_captures),
                                ("CR1", CR1), 
                                ("CR2", CR2), 
                                ("v_attenuator", p_att.ask('VOLT?'), 
                                ('wavelength_range', wav)

In [1]:
import snspd


In [2]:
params = snspd.snspd('D:\SNSPD\SNSPD2\SNSPD5\snspd5-1.yaml')

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\QNL\AppData\Local\Temp\ipykernel_29288\3914008023.py:1: SyntaxWarning: invalid escape sequence '\S'
  params = snspd.snspd('D:\SNSPD\SNSPD2\SNSPD5\snspd5-1.yaml')


In [3]:
params.connection1

'single fibre'

In [4]:
params.connection2

'ferrule to ferrule'